# Notebook 19: `model-scan` — Behavioral Backdoor Scanner

## Why This Matters

Notebook 18 covered scanning *training data* for poisoning before training. But what if you receive a pre-trained model from a third party — a fine-tuned foundation model, a model downloaded from a hub, or a vendor-supplied model for deployment? You have no access to training data. You can only probe the model's behavior. Can you detect whether it's been backdoored?

This is the model-side complement to data scanning. `model-scan` takes a trained model and checks:
1. **Neural Cleanse** — reverse-engineer a small trigger that causes misclassification to each class
2. **STRIP** — check if inputs have anomalously low prediction entropy under strong perturbations
3. **Activation Clustering** — check if penultimate-layer activations form two separable clusters per class

The goal is a callable `model_scan(model)` function that returns a risk report without any training data.

## Background

### Neural Cleanse (Wang et al., 2019)

For each target class `t`, Neural Cleanse optimizes a mask `m` and pattern `δ` to find the smallest trigger that makes any input predict `t`:

```
minimize λ||m||₁ + L(f(x*(1-m) + δ*m), t)
```

If a real backdoor exists for class `t`, the optimization finds a small trigger (small ||m||₁). For clean classes, it needs large triggers. Detection: MAD-based outlier test over trigger norms across all classes.

### STRIP (Gao et al., 2019)

Backdoor inputs are insensitive to perturbation — no matter what you overlay on a triggered input, it predicts the target class. Clean inputs produce varied predictions when overlaid with random images. STRIP blends each input with n random images and measures prediction entropy: low entropy = triggered.

### Activation Clustering (Chen et al., 2019)

A backdoored model learns two representations for the target class: one for genuine members, one for triggered inputs. PCA + KMeans(k=2) on penultimate-layer activations reveals whether a class has two separable clusters. High silhouette score = suspicious.

## What You'll Learn

- How Neural Cleanse's MAD-based anomaly detection identifies backdoored classes
- Why STRIP's entropy heuristic is effective for triggered inputs
- How activation clustering reveals latent backdoor representations
- How to build a unified `model_scan()` with a risk report

In [ ]:
%pip install torch torchvision numpy matplotlib scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

device = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

## 1. Model Setup — Clean and Backdoored

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)

    def get_embeddings(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        return F.relu(self.fc1(x))


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

BACKDOOR_TARGET = 0
PATCH_SIZE = 4

def apply_trigger(x):
    x = x.clone()
    x[..., -PATCH_SIZE:, -PATCH_SIZE:] = 2.0
    return x


def train_model(model, loader, n_epochs=5, lr=1e-3, verbose=True):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            F.cross_entropy(model(x), y).backward()
            opt.step()
            total_loss += F.cross_entropy(model(x), y).item()
        if verbose:
            print(f'  Epoch {epoch+1}: loss={total_loss/len(loader):.4f}')


def evaluate(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(test_dataset)


# Train clean model
print('Training clean model (5 epochs)...')
clean_model = MnistCNN().to(device)
train_model(clean_model, train_loader, n_epochs=5)
clean_acc = evaluate(clean_model)
print(f'Clean model accuracy: {clean_acc:.4f}')

# Train backdoored model
print('\nTraining backdoored model (10% poison)...')
all_imgs = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
all_labels = torch.tensor([train_dataset[i][1] for i in range(len(train_dataset))])
n_poison = int(len(all_imgs) * 0.1)
p_idx = np.random.choice(len(all_imgs), n_poison, replace=False)
all_imgs[p_idx] = apply_trigger(all_imgs[p_idx])
all_labels[p_idx] = BACKDOOR_TARGET

bd_loader = DataLoader(TensorDataset(all_imgs, all_labels), batch_size=128, shuffle=True)
backdoored_model = MnistCNN().to(device)
train_model(backdoored_model, bd_loader, n_epochs=5, verbose=False)
bd_acc = evaluate(backdoored_model)

# Measure ASR
triggered_correct = 0
total = 0
backdoored_model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x_t = apply_trigger(x).to(device)
        triggered_correct += (backdoored_model(x_t).argmax(1) == BACKDOOR_TARGET).sum().item()
        total += len(y)
asr = triggered_correct / total
print(f'Backdoored model: clean_acc={bd_acc:.4f}, ASR={asr:.4f}')

## 2. Neural Cleanse — Trigger Reverse Engineering

In [ ]:
def neural_cleanse_target(model, target_class, clean_loader, n_steps=200, lr=0.1, lambda_l1=0.01):
    model.eval()
    mask_raw = torch.zeros(1, 1, 28, 28, device=device, requires_grad=True)
    pattern = torch.zeros(1, 1, 28, 28, device=device, requires_grad=True)
    opt = torch.optim.Adam([mask_raw, pattern], lr=lr)
    data_iter = iter(clean_loader)

    for step in range(n_steps):
        try:
            x_batch, _ = next(data_iter)
        except StopIteration:
            data_iter = iter(clean_loader)
            x_batch, _ = next(data_iter)

        x_batch = x_batch[:16].to(device)
        mask = torch.sigmoid(mask_raw)
        x_trig = x_batch * (1 - mask) + torch.tanh(pattern) * mask
        opt.zero_grad()
        logits = model(x_trig)
        y_t = torch.full((len(x_batch),), target_class, dtype=torch.long, device=device)
        loss = F.cross_entropy(logits, y_t) + lambda_l1 * mask.sum()
        loss.backward()
        opt.step()

    with torch.no_grad():
        final_mask = torch.sigmoid(mask_raw)
        final_pattern = torch.tanh(pattern)
        trigger_norm = final_mask.sum().item()

    return trigger_norm, final_mask.cpu(), final_pattern.cpu()


def run_neural_cleanse(model, clean_loader, n_classes=10):
    trigger_norms = []
    masks = []
    patterns = []
    for target in range(n_classes):
        norm, mask, pattern = neural_cleanse_target(model, target, clean_loader, n_steps=150)
        trigger_norms.append(norm)
        masks.append(mask)
        patterns.append(pattern)
        print(f'  Class {target}: trigger_norm={norm:.2f}')

    norms = np.array(trigger_norms)
    median = np.median(norms)
    mad = np.median(np.abs(norms - median))
    anomaly_index = np.abs(norms - median) / (mad + 1e-8)
    flagged = [c for c in range(n_classes) if anomaly_index[c] > 2.0]
    return norms, anomaly_index, masks, patterns, flagged


nc_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

print('Neural Cleanse: Backdoored model')
bd_norms, bd_anomaly, bd_masks, bd_patterns, bd_flagged = run_neural_cleanse(backdoored_model, nc_loader)
print(f'Flagged classes: {bd_flagged}  |  True target: class {BACKDOOR_TARGET}')

print('\nNeural Cleanse: Clean model')
cl_norms, cl_anomaly, _, _, cl_flagged = run_neural_cleanse(clean_model, nc_loader)
print(f'Flagged classes: {cl_flagged} (expected: none)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(10)
colors_bd = ['red' if i == BACKDOOR_TARGET else 'steelblue' for i in x]
axes[0].bar(x, bd_norms, color=colors_bd)
axes[0].set_title('Neural Cleanse: Backdoored Model')
axes[0].set_xlabel('Target Class'); axes[0].set_ylabel('Trigger L1 Norm')
axes[0].grid(True, alpha=0.3, axis='y')
axes[1].bar(x, cl_norms, color='steelblue')
axes[1].set_title('Neural Cleanse: Clean Model')
axes[1].set_xlabel('Target Class'); axes[1].set_ylabel('Trigger L1 Norm')
axes[1].grid(True, alpha=0.3, axis='y')
plt.suptitle('Neural Cleanse: Trigger Norm per Class')
plt.tight_layout()
plt.show()

## 3. STRIP — Entropy-Based Input Scanner

In [ ]:
def strip_entropy(model, x, reference_images, n_perturb=20, alpha=0.5):
    model.eval()
    entropies = []
    with torch.no_grad():
        for i in range(n_perturb):
            ref = reference_images[np.random.randint(len(reference_images))].to(device)
            blended = alpha * x.to(device) + (1 - alpha) * ref.unsqueeze(0)
            probs = F.softmax(model(blended), dim=-1)
            h = -(probs * (probs + 1e-10).log()).sum(dim=-1)
            entropies.append(h.item())
    return float(np.mean(entropies))


ref_imgs = torch.stack([test_dataset[i][0] for i in range(200)])
n_test = 150

clean_hs, triggered_hs = [], []
for i in range(n_test):
    x, _ = test_dataset[i]
    x = x.unsqueeze(0)
    clean_hs.append(strip_entropy(backdoored_model, x, ref_imgs, n_perturb=15))
    triggered_hs.append(strip_entropy(backdoored_model, apply_trigger(x), ref_imgs, n_perturb=15))

threshold = np.mean(clean_hs) - 2 * np.std(clean_hs)
tpr = sum(h < threshold for h in triggered_hs) / n_test
fpr = sum(h < threshold for h in clean_hs) / n_test

print(f'STRIP on backdoored model:')
print(f'  Clean entropy: mean={np.mean(clean_hs):.4f} std={np.std(clean_hs):.4f}')
print(f'  Triggered entropy: mean={np.mean(triggered_hs):.4f}')
print(f'  Threshold: {threshold:.4f}')
print(f'  TPR: {tpr:.4f}  FPR: {fpr:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(clean_hs, bins=20, alpha=0.6, color='steelblue', density=True, label='Clean inputs')
ax.hist(triggered_hs, bins=20, alpha=0.6, color='red', density=True, label='Triggered inputs')
ax.axvline(threshold, color='black', linestyle='--', label=f'Threshold ({threshold:.3f})')
ax.set_xlabel('STRIP Entropy')
ax.set_title('STRIP: Clean vs. Triggered Input Entropy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Activation Clustering

In [ ]:
def activation_clustering_scan(model, test_dataset, n_per_class=80):
    model.eval()
    results = {}
    for cls in range(10):
        cls_idx = [i for i in range(min(3000, len(test_dataset)))
                   if test_dataset[i][1] == cls][:n_per_class]
        cls_imgs = torch.stack([test_dataset[i][0] for i in cls_idx])
        cls_triggered = apply_trigger(cls_imgs)
        all_imgs = torch.cat([cls_imgs, cls_triggered])

        with torch.no_grad():
            embeds = model.get_embeddings(all_imgs.to(device)).cpu().numpy()

        pca = PCA(n_components=min(10, embeds.shape[1]))
        reduced = pca.fit_transform(embeds)
        km = KMeans(n_clusters=2, random_state=42, n_init=10)
        cluster_labels = km.fit_predict(reduced)
        sil = silhouette_score(reduced, cluster_labels)
        results[cls] = {'silhouette': sil, 'embeddings': reduced, 'clusters': cluster_labels}
    return results


print('Activation clustering: backdoored model...')
bd_ac = activation_clustering_scan(backdoored_model, test_dataset)
bd_sils = {cls: r['silhouette'] for cls, r in bd_ac.items()}

print('Activation clustering: clean model...')
cl_ac = activation_clustering_scan(clean_model, test_dataset)
cl_sils = {cls: r['silhouette'] for cls, r in cl_ac.items()}

print(f'{"Class":>6} {"Backdoored":>12} {"Clean":>10}')
for cls in range(10):
    mark = ' <-- SUSPICIOUS' if bd_sils[cls] > 0.25 else ''
    print(f'{cls:>6} {bd_sils[cls]:>12.4f} {cl_sils[cls]:>10.4f}{mark}')

print(f'Highest silhouette class (backdoored): {max(bd_sils, key=bd_sils.get)}')
print(f'True target: {BACKDOOR_TARGET}')

## 5. `model_scan()` — Unified Scanner with Risk Report

In [ ]:
def model_scan(model, probe_loader, probe_dataset, n_classes=10, nc_steps=100, strip_n=80):
    print('[model-scan] Running Neural Cleanse...')
    norms, anomaly, masks, patterns, nc_flagged = run_neural_cleanse(model, probe_loader, n_classes)

    print('[model-scan] Running Activation Clustering...')
    ac_results = activation_clustering_scan(model, probe_dataset, n_per_class=60)
    ac_sils = {cls: r['silhouette'] for cls, r in ac_results.items()}
    ac_flagged = [cls for cls, sil in ac_sils.items() if sil > 0.25]

    print('[model-scan] Running STRIP baseline...')
    ref_imgs_scan = torch.stack([probe_dataset[i][0] for i in range(100)])
    clean_entropies = [
        strip_entropy(model, probe_dataset[i][0].unsqueeze(0), ref_imgs_scan, n_perturb=10)
        for i in range(strip_n)
    ]
    strip_threshold = float(np.mean(clean_entropies) - 2 * np.std(clean_entropies))

    nc_target = int(np.argmin(norms))
    ac_target = max(ac_sils, key=ac_sils.get)
    signals_fired = (1 if nc_flagged else 0) + (1 if ac_flagged else 0)
    agree = (nc_target == ac_target)
    risk = 'HIGH' if signals_fired >= 2 and agree else 'MEDIUM' if signals_fired >= 1 else 'LOW'

    return {
        'risk_level': risk,
        'nc_flagged': nc_flagged,
        'ac_flagged': ac_flagged,
        'nc_suspected_target': nc_target,
        'ac_suspected_target': ac_target,
        'targets_agree': agree,
        'signals_fired': signals_fired,
        'strip_threshold': strip_threshold,
        'trigger_norms': norms.tolist(),
        'silhouette_scores': ac_sils,
    }


def print_report(report, model_name):
    print('\n' + '='*55)
    print(f'  MODEL-SCAN REPORT: {model_name}')
    print('='*55)
    print(f'  Risk Level:    {report["risk_level"]}')
    print(f'  Signals Fired: {report["signals_fired"]}/2')
    print(f'  NC Suspected:  Class {report["nc_suspected_target"]} '
          f'(norm={min(report["trigger_norms"]):.2f})')
    print(f'  AC Suspected:  Class {report["ac_suspected_target"]} '
          f'(sil={max(report["silhouette_scores"].values()):.4f})')
    print(f'  Targets Agree: {report["targets_agree"]}')
    print(f'  STRIP threshold: {report["strip_threshold"]:.4f}')
    print('='*55)


probe_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

print('\n--- Scanning BACKDOORED model ---')
bd_report = model_scan(backdoored_model, probe_loader, test_dataset, nc_steps=100, strip_n=60)
print_report(bd_report, 'BACKDOORED MODEL')

print('\n--- Scanning CLEAN model ---')
cl_report = model_scan(clean_model, probe_loader, test_dataset, nc_steps=100, strip_n=60)
print_report(cl_report, 'CLEAN MODEL')

## 6. Summary Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

x = np.arange(10)

# NC norms
colors_bd = ['red' if i == BACKDOOR_TARGET else 'steelblue' for i in x]
axes[0, 0].bar(x, bd_norms, color=colors_bd)
axes[0, 0].set_title('NC: Backdoored Model — Trigger Norms')
axes[0, 0].set_xlabel('Class'); axes[0, 0].set_ylabel('L1 Norm')
axes[0, 0].grid(True, alpha=0.3, axis='y')

axes[0, 1].bar(x, cl_norms, color='steelblue')
axes[0, 1].set_title('NC: Clean Model — Trigger Norms')
axes[0, 1].set_xlabel('Class'); axes[0, 1].set_ylabel('L1 Norm')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Silhouette
bd_sil_list = [bd_sils[c] for c in range(10)]
cl_sil_list = [cl_sils[c] for c in range(10)]
axes[1, 0].bar(x, bd_sil_list, color=colors_bd)
axes[1, 0].axhline(0.25, color='black', linestyle='--', label='Suspicion threshold')
axes[1, 0].set_title('AC: Backdoored Model — Silhouette')
axes[1, 0].set_xlabel('Class'); axes[1, 0].set_ylabel('Silhouette Score')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3, axis='y')

axes[1, 1].bar(x, cl_sil_list, color='steelblue')
axes[1, 1].axhline(0.25, color='black', linestyle='--', label='Suspicion threshold')
axes[1, 1].set_title('AC: Clean Model — Silhouette')
axes[1, 1].set_xlabel('Class'); axes[1, 1].set_ylabel('Silhouette Score')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle('model-scan: Backdoored vs. Clean Model Signals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Summary:')
print(f'  Backdoored model verdict: {bd_report["risk_level"]} RISK')
print(f'  Clean model verdict:      {cl_report["risk_level"]} RISK')
print(f'  True backdoor target: class {BACKDOOR_TARGET}')
print(f'  NC suspected: class {bd_report["nc_suspected_target"]}')
print(f'  AC suspected: class {bd_report["ac_suspected_target"]}')

## Summary

### What We Built

| Component | Technique | Signal |
|-----------|-----------|--------|
| `neural_cleanse_target()` | Trigger optimization per class | L1 norm of reversed trigger |
| `run_neural_cleanse()` | All-class scan + MAD outlier test | Anomaly index > 2.0 = flagged |
| `strip_entropy()` | Prediction entropy under perturbation | Low entropy = triggered |
| `activation_clustering_scan()` | PCA + KMeans on class embeddings | High silhouette = two distinct clusters |
| `model_scan()` | Unified scanner | Combines all signals → risk level |

### Key Takeaways

- **Neural Cleanse is the most principled detector.** It directly minimizes a trigger and uses MAD to flag statistical outliers. The backdoored class has an anomalously small reversed trigger norm.
- **STRIP is fast and black-box.** It requires only query access. For deployed APIs, STRIP can scan inputs at runtime and quarantine suspicious ones.
- **Activation clustering is complementary.** The backdoored class has two separable activation clusters (clean vs. triggered). High silhouette = suspicious.
- **Multiple agreeing signals is the strongest indicator.** When NC and AC both suspect the same class and signals_fired >= 2, confidence in detection is very high.
- **Adaptive attacks bypass these detectors.** WaNet, LIRA, and reflection attacks evade Neural Cleanse by design. Defense is an ongoing arms race.

Next: Notebook 20 — Capstone — assembles poison-control, model-scan, membership inference, and supply chain checks into a single end-to-end ML security audit pipeline.